GOOGLE DRIVE CONNECTED TO COLAB FOR EASILY ACCESSING DATASET



In [33]:

from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


IMPORTANT NECCESSARY LIBRARIES FOR CLEANING


In [44]:
from numpy.random.mtrand import random_sample
import nltk
import pandas as pd
import re
from nltk.corpus import stopwords

LOADED CSV FILES WITH THREE IMPORTANT COLOUMS ACCORDING TO REQUIREMENT AND SAMPLED TO 200K ROWS RANDOMLY.

In [34]:

file_path = "/content/drive/MyDrive/worlddataset/comments.csv"
df = pd.read_csv(
    file_path,
    usecols=["body", "created_utc", "subreddit"]
)
df = df.sample(n = 200000, random_state=42)
df.reset_index(drop=True, inplace=True)
print(df.shape)
df.head()


(200000, 3)


,subreddit,body,created_utc
0,worldnews,> and overnight the narrative was accepted. N...,2025-12-15 15:10:46
1,worldnews,"U are away your taxes don’t pay for anything, ...",2025-01-21 17:28:22
2,worldnews,That’s so incredibly stupid. During ww2 Europe...,2025-03-05 03:46:40
3,worldnews,> The source said the Canadian Security Intell...,2025-03-26 00:37:25
4,worldnews,We make our own,2025-03-08 08:39:05


SAMPLED DATA SAVED

In [35]:
df.to_csv(
    "/content/drive/MyDrive/worlddataset/sample_200k.csv",
    index=False
)

DATE RANGE CHECKED AND CREATED "year", "month", and "day" COLUMNS. THIS WILL LATER HELP IN TIME SERIES

In [36]:

df["date"] = pd.to_datetime(
    df["created_utc"]
)
print("Start:", df["date"].min())
print("End:", df["date"].max())
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
print(df.columns)
df['created_utc'].head(5)

Start: 2025-01-01 03:19:05
End: 2026-02-28 23:26:53
Index(['subreddit', 'body', 'created_utc', 'date', 'year', 'month', 'day'], dtype='object')


,created_utc
0,2025-12-15 15:10:46
1,2025-01-21 17:28:22
2,2025-03-05 03:46:40
3,2025-03-26 00:37:25
4,2025-03-08 08:39:05


1. CONVERTED COMMENTS TEXT BODY TO LOWERCASE
2. REMOVED URLs
3. REMOVED SPECIAL CHARACTERS
4. REOMOVED STOPWORDS

In [54]:

nltk.download('stopwords')


stop_words = set(stopwords.words('english'))

# Start fresh from original column
df['clean_text'] = df['body'].apply(lambda x: str(x).lower())                            # Step 1: lowercase
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r'http\S+|www\S+', '', x))   # Step 2: remove URLs
df['clean_text'] = df['clean_text'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))         # Step 3: remove special chars
df['clean_text'] = df['clean_text'].apply(lambda x: ' '.join([w for w in x.split() if w not in stop_words]))  # Step 4: stopwords

print(df['clean_text'].head(3))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    overnight narrative accepted wasnt hardly anyo...
1    u away taxes dont pay anything government make...
2    thats incredibly stupid ww europe divided diff...
Name: clean_text, dtype: object


AFTER REMOVING ABOVE THINGS, SOME ROWS BECAME EMPTY SO REMOVED EMPTY ROWS HERE

In [63]:
df = df[df['clean_text'].notna()]
df = df[df['clean_text'].str.strip() != '']
print("Rows after cleaning:", df.shape)

Rows after cleaning: (199216, 9)


SOME COMMENTS WERE REMOVED AND THE REMOVED WORD APPEARED AS A COMMENTS SO THIS WAS REMOVED BECAUSE IT DOES NOT ANY MEANING.

In [64]:
df = df[df['clean_text'] != 'removed']
print("After removing [removed] rows:", df.shape)

After removing [removed] rows: (170666, 9)


DATE CONVERTED TO TIMESTAMP

In [67]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

FINAL CLEANED CSV FILE SAVED

In [69]:
df.to_csv('/content/drive/MyDrive/worlddataset/clean_reddit_data.csv', index=False)
print("Done! Final shape:", df.shape)

Done! Final shape: (170666, 9)
